### Group 24

Shiref Khaled Elhalawany -  221100944

Ahmed Anis Hassan - 221100101 

Karim Ashraf Elsayed - 221100391

Kareem Shaheen - 221101524

# Part 4: Hybrid Recommendation Strategies (Section 9)

## 9.1. Implementation of Hybrid Approaches

We implement three strategies to combine Content-Based (CB) and Collaborative Filtering (CF/SVD).

1.  **Weighted Hybrid**: $Score = \alpha \cdot Score_{CB} + (1-\alpha) \cdot Score_{CF\_norm}$
2.  **Switching Hybrid**: Use CF if ratings $\ge 10$, else CB.
3.  **Cascade Hybrid**: Filter Top-50 with CB, Re-rank Top-10 with CF.

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
import os

RESULTS_DIR = "../results"

def get_user_cb_profile(user_id, df_interactions, item_feature_matrix, item_map):
    user_data = df_interactions[df_interactions['user_id'] == user_id]
    if user_data.empty:
        return None
    
    valid_indices = []
    ratings = []
    
    for _, row in user_data.iterrows():
        i_id = row['item_id']
        if i_id in item_map:
            valid_indices.append(item_map[i_id])
            ratings.append(row['rating'])
            
    if not valid_indices:
        return None
        
    item_vecs = item_feature_matrix[valid_indices]
    ratings_arr = np.array(ratings).reshape(-1, 1)
    
    user_vec = item_vecs.multiply(ratings_arr).sum(axis=0)
    
    total_rating = np.sum(ratings)
    if total_rating > 0:
        user_vec = user_vec / total_rating
        
    return csr_matrix(user_vec)

In [ ]:
def run_weighted_hybrid(target_users, df_interactions, item_feature_matrix, item_map_cb, 
                       svd_u, svd_sigma, svd_vt, svd_user_means, user_map_cf, item_map_cf,
                       df_items_map, alpha_values=[0.3, 0.5, 0.7]):
    
    print("\n--- Running Weighted Hybrid Strategy ---")
    
    all_recs = []
    inv_item_map_cb = {v: k for k, v in item_map_cb.items()}
    
    item_titles = pd.Series(df_items_map.title.values, index=df_items_map.item_id).to_dict()
    
    for uid in target_users:
        
        user_vec_cb = get_user_cb_profile(uid, df_interactions, item_feature_matrix, item_map_cb)
        if user_vec_cb is None:
            continue
            
        cb_scores = item_feature_matrix.dot(user_vec_cb.T).toarray().flatten()
        
        if uid in user_map_cf:
            u_idx = user_map_cf[uid]
            cf_mean = svd_user_means[u_idx]
            u_vec = svd_u[u_idx, :].reshape(1, -1)
            cf_preds = np.dot(np.dot(u_vec, svd_sigma), svd_vt).flatten() + cf_mean
            
            cf_scores_norm = (cf_preds - 1.0) / 4.0
            cf_scores_norm = np.clip(cf_scores_norm, 0.0, 1.0)
        else:
             cf_scores_norm = np.zeros_like(cb_scores)
             
        if len(cb_scores) != len(cf_scores_norm):
            print(f"Warning: Score lengths differ for user {uid}. Skipping Weighted.")
            continue
            
        for alpha in alpha_values:
            final_scores = (alpha * cb_scores) + ((1 - alpha) * cf_scores_norm)
            
            top_indices = np.argsort(final_scores)[::-1][:10]
            
            rank = 1
            for i_idx in top_indices:
                item_id = inv_item_map_cb.get(i_idx, "Unknown")
                score = final_scores[i_idx]
                title = item_titles.get(item_id, "Unknown")
                
                all_recs.append({
                    'User': uid,
                    'Method': f'Weighted (alpha={alpha})',
                    'Rank': rank,
                    'Item_ID': item_id,
                    'Score': round(score, 4),
                    'Title': title
                })
                rank += 1
                
    df_out = pd.DataFrame(all_recs)
    df_out.to_csv(os.path.join(RESULTS_DIR, "hybrid_weighted.csv"), index=False)
    print("Weighted Hybrid results saved.")
    return df_out

In [ ]:
def run_switching_hybrid(target_users, df_interactions, threshold=10,
                         item_feature_matrix=None, item_map_cb=None,
                         svd_u=None, svd_sigma=None, svd_vt=None, svd_user_means=None, user_map_cf=None, item_map_cf=None,
                         df_items_map=None):
    
    print("\n--- Running Switching Hybrid Strategy ---")
    
    all_recs = []
    inv_item_map_cb = {v: k for k, v in item_map_cb.items()}
    item_titles = pd.Series(df_items_map.title.values, index=df_items_map.item_id).to_dict()
    
    user_counts = df_interactions['user_id'].value_counts().to_dict()
    
    for uid in target_users:
        count = user_counts.get(uid, 0)
        method_used = ""
        
        if count >= threshold and uid in user_map_cf:
            method_used = "Switching (CF)"
            u_idx = user_map_cf[uid]
            cf_mean = svd_user_means[u_idx]
            u_vec = svd_u[u_idx, :].reshape(1, -1)
            scores = np.dot(np.dot(u_vec, svd_sigma), svd_vt).flatten() + cf_mean
        else:
            method_used = "Switching (CB)"
            user_vec_cb = get_user_cb_profile(uid, df_interactions, item_feature_matrix, item_map_cb)
            if user_vec_cb is None:
                continue
            scores = item_feature_matrix.dot(user_vec_cb.T).toarray().flatten()
            
        top_indices = np.argsort(scores)[::-1][:10]
        rank = 1
        for i_idx in top_indices:
            item_id = inv_item_map_cb.get(i_idx, "Unknown")
            val = scores[i_idx]
            title = item_titles.get(item_id, "Unknown")
            
            all_recs.append({
                'User': uid,
                'Method': method_used,
                'Rank': rank,
                'Item_ID': item_id,
                'Score': round(val, 4),
                'Title': title
            })
            rank += 1
            
    df_out = pd.DataFrame(all_recs)
    df_out.to_csv(os.path.join(RESULTS_DIR, "hybrid_switching.csv"), index=False)
    print("Switching Hybrid results saved.")
    return df_out

In [ ]:
def run_cascade_hybrid(target_users, df_interactions,
                      item_feature_matrix, item_map_cb,
                      svd_u, svd_sigma, svd_vt, svd_user_means, user_map_cf, item_map_cf,
                      df_items_map):
    
    print("\n--- Running Cascade Hybrid Strategy ---")
    all_recs = []
    inv_item_map_cb = {v: k for k, v in item_map_cb.items()}
    item_titles = pd.Series(df_items_map.title.values, index=df_items_map.item_id).to_dict()
    
    for uid in target_users:
        user_vec_cb = get_user_cb_profile(uid, df_interactions, item_feature_matrix, item_map_cb)
        if user_vec_cb is None:
            continue
            
        cb_scores = item_feature_matrix.dot(user_vec_cb.T).toarray().flatten()
        top_50_indices = np.argsort(cb_scores)[::-1][:50]
        
        if uid in user_map_cf:
            u_idx = user_map_cf[uid]
            cf_mean = svd_user_means[u_idx]
            u_vec = svd_u[u_idx, :].reshape(1, -1)
            
            
            re_ranked = []
            for i_idx in top_50_indices:

                v_vec = svd_vt[:, i_idx].reshape(-1, 1)
                pred = np.dot(np.dot(u_vec, svd_sigma), v_vec)[0,0] + cf_mean
                re_ranked.append((i_idx, pred))
                
            re_ranked.sort(key=lambda x: x[1], reverse=True)
            final_top_10 = re_ranked[:10]
            
        else:
            final_top_10 = [(idx, cb_scores[idx]) for idx in top_50_indices[:10]]
            
        rank = 1
        for i_idx, score in final_top_10:
            item_id = inv_item_map_cb.get(i_idx, "Unknown")
            title = item_titles.get(item_id, "Unknown")
            all_recs.append({
                'User': uid,
                'Method': 'Cascade (CB->CF)',
                'Rank': rank,
                'Item_ID': item_id,
                'Score': round(score, 4),
                'Title': title
            })
            rank += 1
            
    df_out = pd.DataFrame(all_recs)
    df_out.to_csv(os.path.join(RESULTS_DIR, "hybrid_cascade.csv"), index=False)
    print("Cascade Hybrid results saved.")
    return df_out

In [ ]:
def compare_hybrid_methods():
    print("\n--- 9.2 Comparison and Justification ---")
    
    comparison_text = """
    ## Comparison of Hybrid Methods
    
    | Hybrid Method | Advantages | Disadvantages |
    |---------------|------------|---------------|
    | **Weighted** (alpha=0.5) | Balances relevance (CB) and serendipity (CF). Smooth transition. | Hard to tune alpha. CF scaling issues. |
    | **Switching** | Handles Cold Start perfectly (Uses CB). Optimizes for data-rich users (CF). | Hard threshold (10 ratings) can be abrupt. |
    | **Cascade** | Computationally efficient (CF only ranks subset). High precision. | 'Zero-hit' problem: if CB misses, CF can't recover it. |
    
    ## Choice of Best Hybrid: **Switching Hybrid**
    **Justification**:
    1.  **Cold Start Handling**: Our dataset likely has many users with few ratings. Switching ensures they get decent CB recommendations immediately.
    2.  **Quality Cap**: For heavy users, SVD (Item-Based CF) typically outperforms CB in capturing latent tastes. Switching leverages this.
    3.  **Simplicity**: It avoids the complex normalization and weighting issues of Weighted Hybrid and the 'Zero-hit' risk of Cascade.
    """
    
    print(comparison_text)
    
    with open(os.path.join(RESULTS_DIR, "hybrid_comparison.md"), "w") as f:
        f.write(comparison_text)
        
    return comparison_text

# Part 5: Cold-Start Handling (Section 10)

We demonstrate the robustness of our Hybrid approach by simulating cold-start scenarios.
We compare **Switching Hybrid** vs **Popularity Baseline** for users with minimal history (3, 5, 10 ratings).

In [ ]:
import random

def simulate_cold_start(df_interactions, min_ratings=20, n_users=20, n_ratings_list=[3, 5, 10]):
    print("\n--- Simulating Cold-Start Scenarios ---")
    
    user_counts = df_interactions['user_id'].value_counts()
    eligible_users = user_counts[user_counts >= min_ratings].index.tolist()
    
    if len(eligible_users) > n_users:
        sampled_users = random.sample(eligible_users, n_users)
    else:
        sampled_users = eligible_users
        
    scenarios = {}
    ground_truth = {}
    
    for uid in sampled_users:
        user_data = df_interactions[df_interactions['user_id'] == uid]
        actual_items = set(user_data[user_data['rating'] >= 3.0]['item_id'].values)
        ground_truth[uid] = actual_items
        
        user_scenarios = {}
        for n in n_ratings_list:
            if len(user_data) >= n:
                masked = user_data.sample(n=n, random_state=42)
                user_scenarios[n] = masked
            else:
                user_scenarios[n] = user_data
        scenarios[uid] = user_scenarios
        
    print(f"Selected {len(sampled_users)} users for simulation.")
    return scenarios, ground_truth, sampled_users

In [ ]:
def recommend_popularity(df_interactions, top_k=10):
    pop = df_interactions['item_id'].value_counts().head(top_k).index.tolist()
    return pop

In [ ]:
def evaluate_cold_start_pipelines(scenarios, ground_truth, df_interactions, 
                                  item_feature_matrix, item_map_cb,
                                  svd_u, svd_sigma, svd_vt, svd_user_means, user_map_cf, item_map_cf,
                                  df_items_map
                                 ):
    
    print("\n--- Evaluating Cold-Start Performance ---")
    
    results = []
    
    pop_recs = recommend_popularity(df_interactions, top_k=10)
    
    n_list = [3, 5, 10]
    
    for n in n_list:
        print(f"Processing N={n}...")
        
        hits_hybrid = 0
        hits_pop = 0
        total_users = 0
        
        for uid, user_scenarios in scenarios.items():
            if n not in user_scenarios:
                continue
                
            masked_df = user_scenarios[n]
            truth = ground_truth[uid]
            
            rec_items = []
            if n < 10:
                user_vec = get_user_cb_profile(uid, masked_df, item_feature_matrix, item_map_cb)
                if user_vec is not None:
                    scores = item_feature_matrix.dot(user_vec.T).toarray().flatten()
                    top_i = np.argsort(scores)[::-1][:10]
                    inv_map = {v:k for k,v in item_map_cb.items()}
                    rec_items = [inv_map.get(i, "UNKNOWN") for i in top_i]
            else:
                if uid in user_map_cf:
                    u_idx = user_map_cf[uid]
                    u_vec = svd_u[u_idx, :].reshape(1, -1)
                    cf_preds = np.dot(np.dot(u_vec, svd_sigma), svd_vt).flatten() + svd_user_means[u_idx]
                    top_i = np.argsort(cf_preds)[::-1][:10]
                    inv_map = {v:k for k,v in item_map_cf.items()}
                    rec_items = [inv_map.get(i, "UNKNOWN") for i in top_i]
                else:
                    rec_items = []
            
            training_items = set(masked_df['item_id'].values)
            rec_items = [x for x in rec_items if x not in training_items]
            rec_items = rec_items[:10] 
            
            if any(item in truth for item in rec_items):
                hits_hybrid += 1
                
            pop_final = [x for x in pop_recs if x not in training_items][:10]
            if any(item in truth for item in pop_final):
                hits_pop += 1
                
            total_users += 1
            
        hr_hybrid = hits_hybrid / total_users if total_users > 0 else 0
        hr_pop = hits_pop / total_users if total_users > 0 else 0
        
        results.append({
            'Scenario (N)': n,
            'Algorithm': 'Switching Hybrid',
            'Hit Rate': hr_hybrid
        })
        results.append({
            'Scenario (N)': n,
            'Algorithm': 'Popularity',
            'Hit Rate': hr_pop
        })
        
    df_res = pd.DataFrame(results)
    save_path = os.path.join(RESULTS_DIR, "cold_start_evaluation.csv")
    df_res.to_csv(save_path, index=False)
    print("\nCold-Start Evaluation Complete.")
    print(df_res)
    return df_res

# Part 6: Baseline Comparison (Section 11)

We compare our Best Hybrid System against three baselines:
1.  **Random Recommender**: Lower bound.
2.  **Popularity Recommender**: Non-personalized baseline.
3.  **Pure Content-Based**: Personalized baseline.

Metric: Hit Rate @ 10 using Leave-One-Out (Sampled).

In [ ]:
def recommend_random(all_item_ids, k=10):
    return random.sample(all_item_ids, k)

def evaluate_baselines_comparison(df_interactions, 
                                  item_feature_matrix, item_map_cb,
                                  svd_u, svd_sigma, svd_vt, svd_user_means, user_map_cf, item_map_cf,
                                  df_items_map,
                                  n_test_users=20):
    
    print("\n--- Running Baseline Comparison (Leave-One-Out) ---")
    
    user_counts = df_interactions['user_id'].value_counts()
    eligible = user_counts[user_counts >= 20].index.tolist()
    if len(eligible) > n_test_users:
        test_users = random.sample(eligible, n_test_users)
    else:
        test_users = eligible
        
    all_items = list(df_items_map['item_id'].unique())
    
    hits = {'Random': 0, 'Popularity': 0, 'Pure CB': 0, 'Switching Hybrid': 0}
    
    pop_items_global = recommend_popularity(df_interactions, top_k=200)
    
    for uid in test_users:
        u_data = df_interactions[df_interactions['user_id'] == uid]
        positive = u_data[u_data['rating'] >= 3.0]
        if positive.empty:
            continue
            
        hidden_item = positive.sample(1, random_state=42)['item_id'].values[0]
        
        masked_user_df = u_data[u_data['item_id'] != hidden_item]
        
        rnd_recs = recommend_random(all_items, k=10)
        if hidden_item in rnd_recs:
            hits['Random'] += 1
            
        rated_items_set = set(masked_user_df['item_id'].values)
        pop_recs = [x for x in pop_items_global if x not in rated_items_set][:10]
        if hidden_item in pop_recs:
            hits['Popularity'] += 1
            
        inv_map_cb = {v:k for k,v in item_map_cb.items()}
        user_vec = get_user_cb_profile(uid, masked_user_df, item_feature_matrix, item_map_cb)
        cb_recs = []
        if user_vec is not None:
            scores = item_feature_matrix.dot(user_vec.T).toarray().flatten()
            top_ind = np.argsort(scores)[::-1]
            count = 0
            for idx in top_ind:
                iid = inv_map_cb.get(idx, "Unknown")
                if iid not in rated_items_set:
                    cb_recs.append(iid)
                    count += 1
                if count >= 10:
                    break
        if hidden_item in cb_recs:
            hits['Pure CB'] += 1
            
        
        cf_recs = []
        if uid in user_map_cf:
            u_idx = user_map_cf[uid]
            u_vec = svd_u[u_idx, :].reshape(1, -1)
            preds = np.dot(np.dot(u_vec, svd_sigma), svd_vt).flatten() + svd_user_means[u_idx]
            top_ind = np.argsort(preds)[::-1]
            inv_map_cf = {v:k for k,v in item_map_cf.items()}
            count = 0
            for idx in top_ind:
                iid = inv_map_cf.get(idx, "Unknown")
                if iid not in rated_items_set:
                     cf_recs.append(iid)
                     count += 1
                if count >= 10:
                    break
        
        if hidden_item in cf_recs:
            hits['Switching Hybrid'] += 1

    final_data = []
    n = len(test_users)
    for method, hit_count in hits.items():
        final_data.append({
            'Method': method,
            'Hit Rate @ 10': hit_count / n
        })
    
    df_final = pd.DataFrame(final_data).sort_values(by='Hit Rate @ 10', ascending=False)
    save_path = os.path.join(RESULTS_DIR, "baseline_comparison.csv")
    df_final.to_csv(save_path, index=False)
    print("\nComparison Table:")
    print(df_final)
    return df_final